# Hyperparameter Black-Scholes
## Convergence (Epochs)

In [ ]:
# Standard library imports
import os
import time
from pathlib import Path

# Data manipulation and mathematics
import numpy as np
import pandas as pd
from scipy.stats import norm

# Visualization
import matplotlib.pyplot as plt
from matplotlib import cm

# Deep learning framework (PyTorch)
import torch
import torch.nn as nn
import torch.optim as optim

# Sweep
import itertools
import seaborn as sns

### Colab Setup

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    repo_url = "https://github.com/egil10/fys5429.git"
    repo_dir = "/content/fys5429"

    if not os.path.exists(repo_dir):
        !git clone {repo_url} {repo_dir}
    else:
        !git -C {repo_dir} pull

    os.chdir(os.path.join(repo_dir, "code", "notebooks"))

print(f"Working directory: {os.getcwd()}")

### Paths

In [ ]:
data_path = Path("..") / "data" / "generated" / "bs_collocation.parquet"

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    out_dir = Path("/content/drive/MyDrive/GITHUB-COLAB/fys5429/code/plots/eda")
else:
    out_dir = Path("..") / "plots" / "eda"

out_dir.mkdir(parents=True, exist_ok=True)
print(f"Plots will be saved to: {out_dir}")

In [ ]:
import sys
sys.path.insert(0, "../scripts")
from bspinn import BSPINN
from train_bs import train_pinn

### Global parameters

In [ ]:
# Answer to the universe and everything
torch.manual_seed(42)

# Use GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Adding torch backends
torch.backends.cudnn.benchmark = True

# Option Physics (Analytical Benchmarks)
K = 100.0
r = 0.05
sigma = 0.20
T_max = 1.0
S_max = 300.0

# NN (best from sweep 1: architecture)
HIDDEN_LAYERS = 2
NEURONS_PER_LAYER = 256

# Learning rate (best from sweep 2)
LEARNING_RATE = 5e-4

# PINN Loss Weights (best from sweep 3)
LAMBDA_PDE = 10.0
LAMBDA_IC = 5.0
LAMBDA_BC = 5.0

# Sweep values: epochs
EPOCHS_VALUES = [1000, 2000, 3000, 5000, 7500, 10000]

### Reading Data

In [ ]:
# Check if data exists and has the right S range, otherwise regenerate
if data_path.exists():
    df_all = pd.read_parquet(data_path)
    s_range = df_all['S'].max()
    if s_range < S_max * 0.9:
        print(f"Existing data has S_max={s_range:.0f}, expected ~{S_max:.0f}. Regenerating...")
        data_path.unlink()

if not data_path.exists():
    print(f"Generating collocation data with S_max={S_max}, T_max={T_max}...")

    N_INTERIOR = 2000
    N_BOUNDARY = 500

    S_interior = torch.rand(N_INTERIOR, 1) * S_max
    tau_interior = torch.rand(N_INTERIOR, 1) * T_max

    S_ic_gen = torch.rand(N_BOUNDARY, 1) * S_max
    tau_ic_gen = torch.zeros(N_BOUNDARY, 1)

    S_bc_gen = torch.zeros(N_BOUNDARY, 1)
    tau_bc_gen = torch.rand(N_BOUNDARY, 1) * T_max

    df_all = pd.concat([
        pd.DataFrame({'S': S_interior.numpy().flatten(),
                       'tau': tau_interior.numpy().flatten(),
                       'point_type': 'interior'}),
        pd.DataFrame({'S': S_ic_gen.numpy().flatten(),
                       'tau': tau_ic_gen.numpy().flatten(),
                       'point_type': 'initial_condition'}),
        pd.DataFrame({'S': S_bc_gen.numpy().flatten(),
                       'tau': tau_bc_gen.numpy().flatten(),
                       'point_type': 'boundary_condition'}),
    ], ignore_index=True)

    data_path.parent.mkdir(parents=True, exist_ok=True)
    df_all.to_parquet(data_path)
    print(f"Saved to {data_path}")
else:
    print(f"Loaded existing data from {data_path} (S_max={df_all['S'].max():.0f})")

# Extract tensors
def extract_tensors(df_subset):
    S_tensor = torch.tensor(df_subset['S'].values, dtype=torch.float32).view(-1, 1).to(device).requires_grad_(True)
    tau_tensor = torch.tensor(df_subset['tau'].values, dtype=torch.float32).view(-1, 1).to(device).requires_grad_(True)
    return S_tensor, tau_tensor

df_interior = df_all[df_all['point_type'] == 'interior']
S_in, tau_in = extract_tensors(df_interior)

df_ic = df_all[df_all['point_type'] == 'initial_condition']
S_ic, tau_ic = extract_tensors(df_ic)

df_bc = df_all[df_all['point_type'] == 'boundary_condition']
S_bc, tau_bc = extract_tensors(df_bc)

print(f"Interior points: {len(S_in)}")
print(f"IC points:       {len(S_ic)}")
print(f"BC points:       {len(S_bc)}")

### Convergence Sweep

In [ ]:
sweep_results = []
start_time = time.time()
total_runs = len(EPOCHS_VALUES)

header = f"{'#':>3} | {'Epochs':>8} | {'PDE Loss':>12} {'IC Loss':>12} {'BC Loss':>12} | {'Time':>6} {'ETA':>8}"
print(header)
print("─" * len(header))

for i, epochs in enumerate(EPOCHS_VALUES):
    run_start = time.time()
    result = train_pinn(S_in, tau_in, S_ic, tau_ic, S_bc, tau_bc,
                    sigma, r, K, device,
                    LAMBDA_PDE, LAMBDA_IC, LAMBDA_BC, epochs,
                    lr=LEARNING_RATE, hidden_layers=HIDDEN_LAYERS,
                    neurons=NEURONS_PER_LAYER, activation='tanh')

    result['epochs'] = epochs
    sweep_results.append(result)

    run_sec = time.time() - run_start
    total_elapsed = time.time() - start_time
    eta = (total_elapsed / (i + 1)) * (total_runs - i - 1)

    print(f"{i+1:>3} | {epochs:>8} | "
          f"{result['final_pde']:>12.6f} {result['final_ic']:>12.6f} {result['final_bc']:>12.6f} | "
          f"{run_sec:>5.0f}s {eta:>6.0f}s")

elapsed = time.time() - start_time
print("─" * len(header))
print(f"Sweep complete: {len(sweep_results)} runs in {int(elapsed//60)}m {int(elapsed%60):02d}s")

### Results

In [ ]:
df_sweep = pd.DataFrame([{
    'epochs': r['epochs'],
    'pde_loss': r['final_pde'],
    'ic_loss': r['final_ic'],
    'bc_loss': r['final_bc'],
    'total_loss': r['final_total'],
} for r in sweep_results])

df_sweep = df_sweep.sort_values('pde_loss')
print(df_sweep.to_string(index=False))

### Ranked by PDE Loss

In [ ]:
df_ranked = df_sweep.sort_values('pde_loss').reset_index(drop=True)
labels = [f"Epochs={r.epochs:.0f}" for _, r in df_ranked.iterrows()]

fig, ax = plt.subplots(figsize=(14, 5))
colors = plt.cm.viridis(np.linspace(0, 1, len(df_ranked)))
bars = ax.barh(range(len(df_ranked)), df_ranked['pde_loss'], color=colors)
ax.set_yticks(range(len(df_ranked)))
ax.set_yticklabels(labels)
ax.set_xlabel('PDE Residual Loss')
ax.set_title('All Epoch Counts Ranked by PDE Loss')
ax.invert_yaxis()

for i, v in enumerate(df_ranked['pde_loss']):
    ax.text(v, i, f'  {v:.2e}', va='center', fontsize=8)

plt.tight_layout()
plt.savefig(out_dir / "hyper_bs_conv_ranked.pdf", bbox_inches="tight")
plt.show()

### Loss Curves (All Runs)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

for r in sweep_results:
    label = f"Epochs={r['epochs']}"
    ax.plot(r['history']['epoch'], r['history']['pde'], label=label, lw=1.2)

ax.set_yscale('log')
ax.set_xlabel('Epoch')
ax.set_ylabel('PDE Loss (Log Scale)')
ax.set_title('PDE Loss Convergence Across Epoch Budgets')
ax.legend()
ax.grid(True, which='both', ls='-', alpha=0.2)
plt.tight_layout()
plt.savefig(out_dir / "hyper_bs_conv_curves.pdf", bbox_inches="tight")
plt.show()

### Download the plots again from Colab

In [ ]:
if IN_COLAB:
    print(f"Plots already saved to Google Drive: {out_dir}")
else:
    print(f"PDFs saved locally to: {out_dir.resolve()}")